# SAGE Phase 18 — overnight fixed-step completion

This notebook has one job: restore the verified partial Phase 18 bundle, run held-out seeds 12–15 concurrently on four CPU processes, merge the 30 fixed-step rows, and export the final evidence bundle.

Before **Save Version → Run All**: add `SAGE_Phase18_partial_20261120_173548.zip` (or its extracted folder) as a Kaggle Input, enable Internet, and use **No accelerator**. Do not change the run name, seeds, steps, methods, or selected settings. Each seed has its own atomic checkpoints, so rerunning this notebook resumes instead of restarting.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, tarfile, time, urllib.request, zipfile

REPO_URL = 'https://github.com/ghugn/SAGE_ICML2027.git'
REPO_BRANCH = 'main'
JULIA_VERSION = '1.13.3'
RUN_NAME = 'phase18_external_v3_seeds_11_15'
REMAINING_SEEDS = (12, 13, 14, 15)
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO = WORK / 'SAGE_ICML2027'
DEPOT = WORK / 'julia_depot'
RUN_DIR = WORK / 'sage_reproduction_logs' / RUN_NAME
RESULT_DIR = RUN_DIR / 'phase18_external_results'

assert WORK.exists(), 'This notebook is intended for Kaggle.'
DEPOT.mkdir(parents=True, exist_ok=True)

# Clone or fast-forward the public code.
if (REPO / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
runner_text = (REPO / 'scripts' / 'benchmark_phase18_external_baselines.jl').read_text(encoding='utf-8')
assert 'confirm_seed' in runner_text, 'Repository does not contain the fixed-step seed runner.'
print('Code commit:', commit)

# Restore the partial bundle only when the working checkpoint is absent.
selection = RESULT_DIR / 'phase18_selection.jls'
if not selection.exists():
    candidates = list(INPUT.rglob(RUN_NAME)) if INPUT.exists() else []
    if not candidates:
        archives = sorted(INPUT.rglob('SAGE_Phase18_partial*.zip')) if INPUT.exists() else []
        assert archives, 'Attach the Phase 18 partial ZIP/folder as a Kaggle Input before Run All.'
        restore_root = WORK / 'phase18_restore'
        if restore_root.exists():
            shutil.rmtree(restore_root)
        restore_root.mkdir(parents=True)
        with zipfile.ZipFile(archives[-1], 'r') as bundle:
            bundle.extractall(restore_root)
        candidates = list(restore_root.rglob(RUN_NAME))
    assert candidates, f'Could not locate {RUN_NAME} inside the attached input.'
    source = candidates[0]
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, RUN_DIR, dirs_exist_ok=True)
    print('Restored checkpoint from:', source)
else:
    print('Using existing working checkpoint:', RUN_DIR)

assert selection.exists(), 'Frozen Phase 18 selection is missing after restore.'
fixed11 = sorted((RESULT_DIR / 'completed_methods').glob('seed_11_*_fixed.jls'))
assert len(fixed11) == 6, f'Expected six complete seed-11 fixed rows, found {len(fixed11)}.'
print('Verified seed-11 fixed checkpoints:', [p.name for p in fixed11])

# Install Julia once.
JULIA = WORK / f'julia-{JULIA_VERSION}' / 'bin' / 'julia'
if not JULIA.exists():
    archive = WORK / f'julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
    url = f'https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, 'r:gz') as bundle:
        bundle.extractall(WORK)
assert JULIA.exists(), JULIA

base_env = os.environ.copy()
base_env.update({
    'JULIA_DEPOT_PATH': str(DEPOT),
    'JULIA_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
})
subprocess.run([str(JULIA), '--threads=1', '--project=.', '-e', 'using Pkg; Pkg.instantiate(); Pkg.precompile()'], cwd=REPO, env=base_env, check=True)
print('OVERNIGHT_SETUP=PASS')

## Four independent held-out seeds

One Julia process is assigned to each remaining seed. Each process uses one CPU thread, avoiding nested oversubscription while using all four Kaggle CPU cores across seeds. Processes share only read-only selection/pilot files; training states and completed-method caches have seed-specific atomic filenames. Derived CSV/JSON reporting is deferred until the single merge step.

In [ ]:
processes = {}
handles = {}
for seed in REMAINING_SEEDS:
    log_path = RUN_DIR / f'overnight_seed_{seed}.log'
    handle = log_path.open('a', encoding='utf-8', buffering=1)
    env = base_env.copy()
    env.update({
        'SAGE_PHASE18_OUTPUT': str(RESULT_DIR),
        'SAGE_PHASE18_PROFILE': 'confirm_seed',
        'SAGE_PHASE18_SEED': str(seed),
    })
    process = subprocess.Popen(
        [str(JULIA), '--threads=1', '--project=.', 'scripts/benchmark_phase18_external_baselines.jl'],
        cwd=REPO, env=env, stdout=handle, stderr=subprocess.STDOUT, text=True)
    processes[seed] = process
    handles[seed] = handle
    print(f'Started seed {seed}: PID {process.pid}, log={log_path}')

last_report = 0.0
while any(process.poll() is None for process in processes.values()):
    time.sleep(30)
    if time.time() - last_report >= 120:
        status = {seed: ('running' if process.poll() is None else process.returncode) for seed, process in processes.items()}
        completed = len(list((RESULT_DIR / 'completed_methods').glob('seed_1[2-5]_*_fixed.jls')))
        print(time.strftime('%H:%M:%S'), 'status=', status, 'remaining fixed rows completed=', completed, '/24', flush=True)
        last_report = time.time()

for handle in handles.values():
    handle.close()
failed = {seed: process.returncode for seed, process in processes.items() if process.returncode != 0}
if failed:
    for seed in failed:
        log_path = RUN_DIR / f'overnight_seed_{seed}.log'
        print(f'--- tail seed {seed} ---')
        print(''.join(log_path.read_text(encoding='utf-8', errors='replace').splitlines(True)[-30:]))
    raise RuntimeError(f'Seed processes failed: {failed}. Atomic checkpoints are preserved; rerun unchanged to resume.')
print('PHASE18_OVERNIGHT_SEEDS=COMPLETE')

## Strict 30-row merge and export

The merge refuses to complete unless all six methods exist for all five registered seeds.

In [ ]:
merge_env = base_env.copy()
merge_env.update({
    'JULIA_NUM_THREADS': '4',
    'SAGE_PHASE18_OUTPUT': str(RESULT_DIR),
    'SAGE_PHASE18_PROFILE': 'merge',
})
merge_log = RUN_DIR / 'overnight_merge.log'
with merge_log.open('w', encoding='utf-8') as handle:
    completed = subprocess.run(
        [str(JULIA), '--threads=4', '--project=.', 'scripts/benchmark_phase18_external_baselines.jl'],
        cwd=REPO, env=merge_env, stdout=handle, stderr=subprocess.STDOUT, text=True)
merge_text = merge_log.read_text(encoding='utf-8', errors='replace')
print(merge_text)
assert completed.returncode == 0 and 'PHASE18_EXTERNAL_CONFIRMATORY=COMPLETE' in merge_text, 'Strict merge failed; inspect overnight_merge.log.'

summary_path = RESULT_DIR / 'phase18_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert summary['status'] == 'COMPLETE'
assert summary['protocol'] == 'fixed_1024_updates'

manifest = {
    'status': 'PHASE18_FIXED_COMPLETE',
    'git_commit': commit,
    'summary_sha256': hashlib.sha256(summary_path.read_bytes()).hexdigest(),
    'fixed_rows': 30,
    'protocol': 'fixed_1024_updates',
}
(RUN_DIR / 'OVERNIGHT_MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = Path(shutil.make_archive(str(WORK / 'SAGE_Phase18_fixed_complete'), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
print(json.dumps(summary, indent=2))
print('Final archive:', archive, f'({archive.stat().st_size / 1024**2:.2f} MB)')
print('Download it from the saved Kaggle Version Output tab.')
print('PHASE18_OVERNIGHT=COMPLETE')